In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

In [ ]:
# Define the path to the processed data based on the current user
current_user = os.getlogin()

# Define output directory (notebook is in ./notebooks; data is ../data)
DATA_DIR     = Path(rf"/home/{current_user}/local-share")
OUT_DIR      = DATA_DIR / "processed"

missing_values_path = OUT_DIR / "features_standardized.csv"

# Let pandas detect the delimiter
data = pd.read_csv(missing_values_path, sep=None, engine="python", encoding="utf-8-sig")
print(data.shape)
data

In [ ]:
data.columns

In [ ]:
# Look at raw examples
data['sdo2_orientation_Event_Types_Attended'].dropna().head(20)


In [ ]:
# -------------------------------------------------------------------------
# PURPOSE
# -------------------------------------------------------------------------
# Inspect and understand the true event categories contained in
# 'sdo2_orientation_Event_Types_Attended'.
#
# This step:
#   • Converts the column to clean lowercase text
#   • Splits multi-event strings into individual event entries
#     (so "open dag, proefstuderen" becomes two rows)
#   • Removes extra spaces and normalizes formatting
#   • Extracts and prints the list of unique event categories
#   • Displays frequency counts for each category
#
# Outcome:
#   Provides a clean overview of all event types present in the data,
#   confirming the categories required for correct multi-hot encoding.
# -------------------------------------------------------------------------

# Split, explode, and clean
event_series = (
    data['sdo2_orientation_Event_Types_Attended']
      .dropna()
      .astype(str)
      .str.lower()
      .str.split(r'\s*,\s*')   # split on commas, trim spaces
      .explode()
      .str.strip()
)

# See the true unique categories
unique_events = sorted(event_series.unique())
print(unique_events)
print("n unique:", len(unique_events))

# Count frequency of each category
print(event_series.value_counts())


In [ ]:
# -------------------------------------------------------------------------
# PURPOSE
# -------------------------------------------------------------------------
# Extract and inspect unique event types from 
# 'sdo2_orientation_Event_Types_Attended'.
#
# This step:
#   • Cleans and normalizes the text (lowercase, trim spaces)
#   • Applies controlled renaming for safe category names:
#         "online opleidingspresentatie" → "online_opleidingspresentatie"
#         "open dag" → "open_dag"
#   • Splits multi-event strings into separate rows (explode)
#   • Prints the list of unique event categories and their frequencies
#
# Outcome:
#   Ensures consistent, model-friendly category names and verifies the
#   total set of event types before multi-hot encoding.
# -------------------------------------------------------------------------

# One-step cleaning + renaming + exploding + printing
event_series = (
    data['sdo2_orientation_Event_Types_Attended']
      .dropna()
      .astype(str)
      .str.lower()
      .str.strip()
      .str.replace("online opleidingspresentatie", "online_opleidingspresentatie", regex=False)
      .str.replace("open dag", "open_dag", regex=False)
      .str.split(r'\s*,\s*')
      .explode()
      .str.strip()
)

# Print results
print("Unique categories:", sorted(event_series.unique()))
print("n unique:", len(event_series.unique()))
print("\nCounts:\n", event_series.value_counts())


# Feature Encoding

In [ ]:
# -------------------------------------------------------------------------
# PURPOSE
# -------------------------------------------------------------------------
# Robust, idempotent multi-hot encoding of event categories from `event_series`
# for `sdo2_orientation_Event_Types_Attended`.
#
# Naming rules (NO column-name prefix added):
#   • "absent"  -> column named exactly "absent"
#   • others    -> "attended_<category>"
#
# Other rules:
#   • Missing values in the new dummy columns are filled with 0.
#   • The original mother column `sdo2_orientation_Event_Types_Attended`
#     is dropped at the end.
#
# Robustness:
#   • Safe to run multiple times.
#   • If expected columns already exist, it skips and reports so.
#   • If partially present, it drops and recreates them cleanly.
# -------------------------------------------------------------------------

# Expected output column names based on current event_series
expected_events = sorted(event_series.unique())
expected_cols = [
    (e if e == "absent" else f"attended_{e}")
    for e in expected_events
]

already_done = all(col in data.columns for col in expected_cols)

if already_done:
    print("✓ One-hot encoding already done. Skipping.")
else:
    # If partially present, drop to avoid collisions and re-create cleanly
    cols_to_drop = [c for c in expected_cols if c in data.columns]
    if cols_to_drop:
        data.drop(columns=cols_to_drop, inplace=True)
        print(f"ℹ️ Partial encoding found; re-creating columns: {cols_to_drop}")

    # 1) Map original row index -> event
    tmp = event_series.reset_index()
    tmp.columns = ["row_id", "event"]

    # 2) Create multi-hot matrix for all categories (including absent)
    multi_hot = pd.crosstab(tmp["row_id"], tmp["event"]).astype("Int64")

    # 3) Rename columns: absent stays "absent", others get "attended_" prefix
    rename_map = {c: (c if c == "absent" else f"attended_{c}") for c in multi_hot.columns}
    multi_hot.rename(columns=rename_map, inplace=True)

    # 4) Join back to main dataframe
    data = data.join(multi_hot, how="left")

    # 5) Fill NaN in the new dummy columns with 0
    new_event_cols = list(multi_hot.columns)
    data[new_event_cols] = data[new_event_cols].fillna(0).astype("Int64")

    print("✓ Created event columns:", new_event_cols)

# 6) Drop mother column at the end (safe even if already dropped)
data.drop(columns=["sdo2_orientation_Event_Types_Attended"], inplace=True, errors="ignore")

# Preview
data[[c for c in expected_cols if c in data.columns]].head()


In [ ]:
data

In [ ]:
# ---------------------------------------------------------------------
# PURPOSE
# ---------------------------------------------------------------------
# One-hot encode remaining categorical columns while ensuring:
#   • Encoded dummy values are integers (0/1)
#   • Original columns are removed
#   • 'sdo2_orientation_Event_Types_Attended' is excluded for now
# ---------------------------------------------------------------------

encode_cols = [
    # 'sdo5_degree_COLLEGEJAAR', <-- excluded intentionally
    'sdo5_degree_degree',
    'sdo1_previous_Previous_Education_Type',
    'sdo2_skc_ADVIES_DEF',
    # 'sdo2_orientation_Event_Types_Attended',  <-- excluded for now
    'age_category'
]

# Perform one-hot encoding
data = pd.get_dummies(
    data,
    columns=encode_cols,
    drop_first=False,
    dtype='int'   # <-- ensures 0/1 integers instead of True/False
)
data

# Split data year-wise , for train/test/validation